# PySpark

## Row object in PySpark

In [0]:
from pyspark.sql import Row

row1 =  Row(name = "Alawie", animal_type = "Fox", age = 9)
print(row1)

In [0]:
row2 = Row(name = "Mohannad", animal_type = "Dog", age = 5)
print(row2)

### Create spark data frame

In [0]:
df = spark.createDataFrame([row1, row2])
df

In [0]:
df.show()

In [0]:
df.take(2)

In [0]:
display(df.take(2))

In [0]:
display(df)

In [0]:
employees = [
    {"name": "Alawie", "age": 30, "department": "IT"},
    {"name": "Mohannad", "age": 25, "department": "HR"},
    {"name": "Rami", "age": 35, "department": "IT"}]

df_employees = spark.createDataFrame(employees)
df_employees

In [0]:
display(df_employees)

In [0]:
df_employees.take(2)

## Read from csv file

In [0]:
from pathlib import Path

DATA_PATH = Path().resolve() / "data"

print(DATA_PATH)

df_athlete = spark.read.csv(str(DATA_PATH / "athlete_events.csv"),header = True)

display(df_athlete.take(5))


In [0]:
df_athlete.printSchema()

In [0]:
print(df_athlete.columns)

In [0]:
from pyspark.sql.types import StructField, StructType, StringType, IntegerType, ByteType, ShortType

schema = StructType([
    StructField("ID", IntegerType(), True),
    StructField("Name", StringType(), True),
    StructField("Sex", StringType(), True),
    StructField("Age", ByteType(), True),
    StructField("Height", ShortType(), True),
    StructField("Weight", ShortType(), True),
    StructField("Team", StringType(), True),
    StructField("NOC", StringType(), True),
    StructField("Games", StringType(), True),
    StructField("Year", ShortType(), True),
    StructField("Season", StringType(), True),
    StructField("City", StringType(), True),
    StructField("Sport", StringType(), True),
    StructField("Event", StringType(), True),
    StructField("Medal", StringType(), True)
])

df_athlete_schema = spark.read.csv(str(DATA_PATH / "athlete_events.csv"), header = True, schema = schema)

display(df_athlete_schema.take(5))

## EDA

- calculate null

In [0]:
from pyspark.sql.functions import col, sum

nulls = df_athlete_schema.select([sum(col(c).isNull().cast("int")).alias(c) for c in df_athlete_schema.columns])

display(nulls)

In [0]:
df_athlete_schema.groupBy("NOC").count().filter(
    "NOC IN ('SWE', 'NOR', 'FIN', 'DEN')"
).show()

### Combine SQL and dataframes

In [0]:
df_athlete_schema.createOrReplaceTempView("df_athlete_schema")

df_swe_med = spark.sql("""
    SELECT
        sport, count(medal) medals
    FROM df_athlete_schema
    WHERE noc = 'SWE' AND medal IN ('Gold', 'Silver', 'Bronze')
    GROUP BY sport
    ORDER BY medals DESC  
""") 

display(df_swe_med.take(10))

In [0]:
fig = df_swe_med.plot(kind = "bar", x = "medals", y = "sport", title = "Swedish medals")

fig.update_layout(yaxis = {"autorange": "reversed"})

## SQL cells

In [0]:
df

In [0]:
%sql
FROM df_athlete_schema LIMIT 1

In [0]:
%sql
SHOW CATALOGS;

DROP CATALOG IF EXISTS data CASCADE;

CREATE CATALOG IF NOT EXISTS data;

CREATE SCHEMA IF NOT EXISTS data.olympics;

CREATE OR REPLACE TABLE data.olympics.sweden_medals AS
(
  SELECT
    name, age, weight, height, year, sport, medal
  FROM
    df_athlete_schema
  WHERE
    noc = 'SWE'
    AND medal IN ('Gold', 'Silver', 'Bronze')
);

FROM
  data.olympics.sweden_medals
LIMIT 5